# 🧹 FashionFind — Data Cleaning Pipeline
**Built by Neeta Singh**

This notebook documents the full EDA and cleaning process on the real Myntra dataset before feeding it into our RAG pipeline.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('myntra202305041052.xlsx')
print(f'Raw dataset: {len(df):,} rows x {len(df.columns)} columns')
print(f'Columns: {df.columns.tolist()}')

## Step 1 — Explore the Raw Data

In [ ]:
# Check null values
print('=== NULL VALUES ===')
print(df.isnull().sum())

print(f'\n=== DUPLICATES ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Price distribution & outliers
print('=== PRICE STATS ===')
print(df['price'].describe())
print(f'\nProducts under ₹50 (suspicious): {len(df[df["price"] < 50])}')
print(f'Products over ₹50,000 (extreme luxury): {len(df[df["price"] > 50000])}')

In [ ]:
# Rating analysis
print('=== RATING STATS ===')
print(df['rating'].describe())
print(f'\nProducts with 0 rating (no reviews): {len(df[df["rating"] == 0]):,}')
print(f'Products with valid ratings: {len(df[df["rating"] > 0]):,}')

In [ ]:
# Top sellers
print('=== TOP 10 SELLERS ===')
print(df['seller'].value_counts().head(10))

print('\n=== ASIN Column Check ===')
print(df['asin'].value_counts().head(3))  # All dashes — useless column!

## Step 2 — Clean the Data

**What we found & why we fix it:**
- `name`: 5 null values → drop them
- `asin`: all `-` dashes → useless, drop column
- `img`: massive URLs we don't need → drop column
- `price`: outliers below ₹50 and above ₹50,000 → remove
- `rating`: 781,192 products with 0 rating → remove (unreliable for recommendations)
- `description`: doesn't exist → we CREATE it from other columns (critical for RAG!)

In [ ]:
# Step 1: Drop null names
df = df[df['name'].notna()].copy()
print(f'After dropping null names: {len(df):,}')

# Step 2: Drop useless columns
df = df.drop(columns=['asin', 'img'])
print('Dropped: asin (all dashes), img (huge URLs)')

# Step 3: Remove price outliers
df = df[(df['price'] >= 50) & (df['price'] <= 50000)]
print(f'After price filter (₹50-₹50k): {len(df):,}')

# Step 4: Keep only rated products
df = df[df['rating'] > 0].copy()
print(f'After keeping rated products: {len(df):,}')

# Step 5: Fix data types
df['name'] = df['name'].astype(str).str.strip()
df['seller'] = df['seller'].astype(str).str.strip()
df['price'] = df['price'].astype(int)
df['rating'] = df['rating'].astype(float).round(1)
df['purl'] = df['purl'].astype(str)

# Step 6: Extract category from Myntra URL
df['category'] = df['purl'].str.extract(r'myntra\.com/([^/]+)/')
print(f'\nCategories found: {df["category"].nunique()}')

# Step 7: CREATE description column for RAG embeddings!
df['description'] = (
    df['name'] + ' by ' +
    df['seller'] +
    '. Price: ₹' + df['price'].astype(str) +
    '. Rating: ' + df['rating'].astype(str) + '/5' +
    '. Category: ' + df['category'].fillna('fashion')
)
print('✅ Description column created for RAG!')

# Step 8: Sample 500 representative products
df_clean = df.sample(n=500, random_state=42).reset_index(drop=True)
df_clean = df_clean[['id','name','category','price','mrp','rating','ratingTotal','discount','seller','purl','description']]
print(f'\n✅ Final clean dataset: {len(df_clean)} rows')

## Step 3 — Validate Clean Data

In [ ]:
print('=== FINAL VALIDATION ===')
print(f'Rows: {len(df_clean)}')
print(f'Null values:\n{df_clean.isnull().sum()}')
print(f'\nPrice range: ₹{df_clean["price"].min()} - ₹{df_clean["price"].max()}')
print(f'Avg rating: {df_clean["rating"].mean():.2f}/5')
print(f'\nTop categories:')
print(df_clean['category'].value_counts().head(10))
print(f'\nSample description:')
print(df_clean['description'].iloc[0])

In [ ]:
# Save clean file
df_clean.to_csv('products_clean.csv', index=False)
print('✅ Saved as products_clean.csv — ready for RAG pipeline!')

## Summary

| Step | Before | After |
|---|---|---|
| Raw rows | 1,048,575 | — |
| Null names removed | — | 1,048,570 |
| Price outliers removed | — | 1,048,335 |
| Unrated products removed | — | 267,327 |
| Final sample | — | **500** |

The clean dataset is lightweight (138 KB), high quality, and ready to power our RAG pipeline! 🚀